In [ ]:
import json
import pandas as pd
import folium
from branca.element import Element
from folium.plugins import HeatMapWithTime

df = pd.read_csv("../data/sf_crime_merged_focus_2003_2025_cleaned.csv")
df["Incident Date"] = pd.to_datetime(df["Incident Date"], errors="coerce")
df["Latitude"] = pd.to_numeric(df["Latitude"], errors="coerce")
df["Longitude"] = pd.to_numeric(df["Longitude"], errors="coerce")

with open("sfpd.geojson", "r") as f:
    sfpd_geo = json.load(f)

drug = df[
    (df["Focus Crime"] == "Drug Offense")
    & df["Incident Date"].notna()
    & df["Latitude"].notna()
    & df["Longitude"].notna()
].copy()
drug["DISTRICT"] = drug["Police District"].astype(str).str.strip().str.upper()
drug = drug[drug["DISTRICT"] != "OUT OF SF"]

drug["year"] = drug["Incident Date"].dt.year.astype(int)
drug = drug[(drug["year"] >= 2003) & (drug["year"] <= 2025)]

years = sorted(drug["year"].unique())
hm_data = []
for year in years:
    chunk = drug.loc[drug["year"] == year, ["Latitude", "Longitude"]].copy()
    weighted = (
        chunk
        .groupby(["Latitude", "Longitude"], as_index=False)
        .size()
        .rename(columns={"size": "count"})
    )
    scale = weighted["count"].quantile(0.95)
    if pd.isna(scale) or scale <= 0:
        scale = weighted["count"].max()
    weighted["weight"] = (weighted["count"] / scale).clip(upper=1.0)
    hm_data.append(weighted[["Latitude", "Longitude", "weight"]].astype(float).values.tolist())

year_summaries = {}
for year in years:
    sub_year = drug[drug["year"] == year]
    total = len(sub_year)
    by_district = sub_year["DISTRICT"].value_counts().head(3)
    top_district = by_district.index[0]
    top_count = int(by_district.iloc[0])
    top_share = top_count / total
    lines = [
        f"{year}",
        # f"  Total incidents: {total:,}",
        # f"  Top district: {top_district} ({top_count:,}, {top_share:.1%} of SF total)",
        "Top 3 districts:",
    ]
    for district, count in by_district.items():
        lines.append(f"{district}: {count:,} ({count/total:.1%})")
    year_summaries[str(year)] = "\n".join(lines)

year_district_stats = {}
for year in years:
    sub_year = drug[drug["year"] == year]
    total = len(sub_year)
    counts = sub_year["DISTRICT"].value_counts()
    year_district_stats[str(year)] = {}
    for district, count in counts.items():
        year_district_stats[str(year)][district] = {
            "incidents": f"{int(count):,}",
            "share": f"{(count / total):.1%}",
        }

print(f"Frames: {len(years)}")
print(f"Total Drug Offense rows used: {len(drug):,}")

m_time = folium.Map(
    location=[37.7749, -122.4194],
    zoom_start=12,
    tiles="OpenStreetMap"
)

district_layer = folium.GeoJson(
    sfpd_geo,
    name="Districts",
    style_function=lambda feature: {
        "fillColor": "#00000000",
        "color": "#2f2f2f",
        "weight": 2,
        "opacity": 0.9,
    },
).add_to(m_time)

HeatMapWithTime(
    hm_data,
    index=[str(year) for year in years],
    auto_play=False,
    min_opacity=0.15,
    max_opacity=0.55,
    radius=11,
    blur=0.6,
    gradient={0.35: 'blue', 0.55: 'lime', 0.75: 'orange', 1.0: 'red'},
    use_local_extrema=False,
    min_speed=1,
    max_speed=4,
    speed_step=0.5,
).add_to(m_time)

summary_box = """
<div id='year-summary-box' style='
    position: fixed;
    top: 16px;
    right: 16px;
    z-index: 9999;
    background: rgba(255, 255, 255, 0.92);
    border: 1px solid #666;
    border-radius: 5px;
    box-shadow: 0 1px 6px rgba(0,0,0,0.18);
    padding: 6px 8px;
    width: 220px;
    max-width: 220px;
    font-family: monospace;
    font-size: 10px;
    line-height: 1.25;
    white-space: pre-wrap;
'></div>
<script>
const yearSummaries = __YEAR_SUMMARIES__;
function updateYearSummary(year) {
  const box = document.getElementById('year-summary-box');
  if (box && yearSummaries[String(year)]) {
    box.textContent = yearSummaries[String(year)];
  }
}
function syncYearSummaryFromLabel() {
  const controls = document.querySelectorAll('.leaflet-control-timecontrol');
  for (const control of controls) {
    const text = control.textContent || '';
    const match = text.match(/\\b(20\\d{2})\\b/);
    if (match) {
      updateYearSummary(match[1]);
      return;
    }
  }
}
setTimeout(function() {
  const map = __MAP_NAME__;
  updateYearSummary('__INITIAL_YEAR__');
  syncYearSummaryFromLabel();
  setInterval(syncYearSummaryFromLabel, 250);
}, 500);
</script>
"""
summary_box = summary_box.replace("__YEAR_SUMMARIES__", json.dumps(year_summaries))
summary_box = summary_box.replace("__MAP_NAME__", m_time.get_name())
summary_box = summary_box.replace("__INITIAL_YEAR__", str(years[0]))
m_time.get_root().html.add_child(Element(summary_box))

district_tooltips = """
<div id='district-hover-box' style='
    position: fixed;
    display: none;
    z-index: 10000;
    pointer-events: none;
    background: rgba(255, 255, 255, 0.96);
    border: 1px solid #666;
    border-radius: 6px;
    box-shadow: 0 2px 8px rgba(0,0,0,0.2);
    padding: 8px 10px;
    font-family: monospace;
    font-size: 12px;
    line-height: 1.4;
'></div>
<script>
const districtStatsByYear = __DISTRICT_STATS__;
const districtLayerName = '__DISTRICT_LAYER__';
let activeYearForDistricts = '__INITIAL_YEAR__';
let hoveredDistrict = null;

function districtTooltipHtml(district) {
  const stats = (districtStatsByYear[activeYearForDistricts] || {})[district] || {incidents: '0', share: '0.0%'};
  return 'District: ' + district + '<br>' +
         'Incidents: ' + stats.incidents + '<br>' +
         'Share of total: ' + stats.share;
}

function getDistrictLayer() {
  return window[districtLayerName];
}

function getHoverBox() {
  return document.getElementById('district-hover-box');
}

function showDistrictBox(district, originalEvent) {
  const box = getHoverBox();
  if (!box) return;
  hoveredDistrict = district;
  box.innerHTML = districtTooltipHtml(district);
  box.style.display = 'block';
  box.style.left = (originalEvent.clientX + 14) + 'px';
  box.style.top = (originalEvent.clientY + 14) + 'px';
}

function hideDistrictBox() {
  const box = getHoverBox();
  if (!box) return;
  hoveredDistrict = null;
  box.style.display = 'none';
}

function moveDistrictBox(originalEvent) {
  const box = getHoverBox();
  if (!box || box.style.display === 'none') return;
  box.style.left = (originalEvent.clientX + 14) + 'px';
  box.style.top = (originalEvent.clientY + 14) + 'px';
}

function bindDistrictHover() {
  const districtLayer = getDistrictLayer();
  if (!districtLayer || !districtLayer.eachLayer) return;
  districtLayer.eachLayer(function(layer) {
    const district = String(layer.feature.properties.DISTRICT || '').trim().toUpperCase();
    layer.off();
    layer.on('mouseover', function(e) {
      showDistrictBox(district, e.originalEvent);
    });
    layer.on('mousemove', function(e) {
      moveDistrictBox(e.originalEvent);
      if (hoveredDistrict === district) {
        getHoverBox().innerHTML = districtTooltipHtml(district);
      }
    });
    layer.on('mouseout', function() {
      hideDistrictBox();
    });
  });
}

function syncDistrictYearFromLabel() {
  const controls = document.querySelectorAll('.leaflet-control-timecontrol');
  for (const control of controls) {
    const text = control.textContent || '';
    const match = text.match(/\\b(20\\d{2})\\b/);
    if (match) {
      activeYearForDistricts = match[1];
      if (hoveredDistrict) {
        getHoverBox().innerHTML = districtTooltipHtml(hoveredDistrict);
      }
      return;
    }
  }
}

setTimeout(function() {
  bindDistrictHover();
  syncDistrictYearFromLabel();
  setInterval(syncDistrictYearFromLabel, 250);
}, 700);
</script>
"""
district_tooltips = district_tooltips.replace("__DISTRICT_STATS__", json.dumps(year_district_stats))
district_tooltips = district_tooltips.replace("__DISTRICT_LAYER__", district_layer.get_name())
district_tooltips = district_tooltips.replace("__INITIAL_YEAR__", str(years[0]))
m_time.get_root().html.add_child(Element(district_tooltips))

m_time.save("drug_offense_heatmap_with_districts_2003_2025.html")
m_time
